# Independent validation of fabric visibility models

Fit the linear and power-sum visibility models using the original dataset, then evaluate them using the new 24-sample combined report.

In Colab, uncomment the two Google Drive lines below and run the cells in order. In local Jupyter, use paths on your computer. The notebook uses NumPy, pandas, SciPy, and openpyxl.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.optimize import least_squares

# from google.colab import drive
# drive.mount("/content/drive")

## 1. Define the original models

These are the model equations and fitting settings from “Fit and cross-validate linear and power-sum models (Claude)” in the main notebook. This section defines the fitting functions. The next section creates the fitted models in memory from the original training report. The original fit workbook stores rounded formula text and does not contain a saved model object.

In [ ]:
RESPONSES = [                    # (column in report, display name)
    ('SSIM',               'SSIM'),
    ('Acutance',           'Acutance'),
    ('Michelson_Contrast', 'Michelson Contrast'),
]


PREDICTORS = [
    ('weighted_T_dir', 'T_dir'),
    ('diffuseness',    'Diffuseness Ratio'),
    ('opacity',        'Opacity (R_b/R_w)'),
]


EXPONENT_BOUNDS = (-5.0, 5.0)   # exponents a', b', c' are searched inside this range


N_RANDOM_STARTS = 20            # extra random starting points for the exponent search


RANDOM_SEED = 0


def fit_linear(Xt, yt):
    A = np.column_stack([np.ones(len(Xt)), Xt])
    coef, *_ = np.linalg.lstsq(A, yt, rcond=None)
    return {
        'k': 4,
        'coef': coef,
        'predict': lambda Xn: np.column_stack([np.ones(len(Xn)), Xn]) @ coef,
    }


def fit_power_sum(Xt, yt):
    m = Xt.mean(axis=0)
    Xs = Xt / m
    lo, hi = EXPONENT_BOUNDS

    def design(e, Z):
        return np.column_stack([Z[:, i] ** e[i] for i in range(Z.shape[1])])

    def solve(e):
        Phi = design(e, Xs)
        c, *_ = np.linalg.lstsq(Phi, yt, rcond=None)
        return c, Phi

    def resid(e):
        c, Phi = solve(e)
        return yt - Phi @ c

    rng = np.random.default_rng(RANDOM_SEED)
    starts = [np.ones(Xt.shape[1])] + [rng.uniform(-3, 3, Xt.shape[1]) for _ in range(N_RANDOM_STARTS)]

    best = None
    for e0 in starts:
        try:
            r = least_squares(resid, np.clip(e0, lo, hi), bounds=(lo, hi))
        except (ValueError, FloatingPointError):
            continue
        if not np.isfinite(r.cost):
            continue
        if best is None or r.cost < best.cost:
            best = r
    if best is None:
        return None

    e = best.x
    c, _ = solve(e)
    return {
        'k': 6,
        'exponents': e,
        'coef': c / m ** e,                      # coefficients in ORIGINAL units
        'at_bound': bool(np.any((np.abs(e - lo) < 1e-3) | (np.abs(e - hi) < 1e-3))),
        'predict': lambda Xn: design(e, Xn / m) @ c,
    }


MODELS = [('Linear', fit_linear), ('Power-sum', fit_power_sum)]

## 2. Set the report paths and run the independent test

- **`TRAIN_REPORT`** is the combined optical–image report for the original 55 samples. In the original workflow this is `All data report/all_report.xlsx`. It contains one row per original sample in the **Per-Sample Values** sheet. If you need to recreate it, run **Combined optical–image report** in the main notebook with the original spectral workbooks and photos. Use the original preprocessing and save the workbook as `all_report.xlsx`.
- **`TEST_REPORT`** is the combined report you already generated for the 24 new samples. It has the same columns and uses the same optical units and image metric settings as the training report.

The notebook uses the optical predictors and measured image metrics from the original report to fit each model at `fit = fit_model(x_train, y_train)`. It then calls `fit["predict"]` on the new optical inputs and uses the new measured image metrics to score the predictions.

The linear model uses all 24 rows with finite inputs and measured image metrics. The power-sum model requires positive optical predictors. When a row is outside that domain, it stays in the output with a linear prediction and a blank power-sum prediction. The test metrics include scores for both models on the same rows with positive predictors.

In [ ]:
# Original 55-sample combined optical–image report.
TRAIN_REPORT = Path("/content/drive/MyDrive/Master/All data report/all_report.xlsx")
# Combined optical–image report generated for the new 24 samples.
TEST_REPORT = Path("/content/drive/MyDrive/Master/New data/24_sample_report.xlsx")
TEST_OUTPUT = TEST_REPORT.with_name("24_sample_model_validation.xlsx")

if TRAIN_REPORT.resolve() == TEST_REPORT.resolve():
    raise ValueError("Use the original training report and a separate 24-sample test report.")
if TEST_OUTPUT.resolve() in {TRAIN_REPORT.resolve(), TEST_REPORT.resolve()}:
    raise ValueError("Choose a separate filename for the validation output.")

if not TRAIN_REPORT.is_file():
    raise FileNotFoundError(f"Set TRAIN_REPORT to the original 55-sample combined report: {TRAIN_REPORT}")
if not TEST_REPORT.is_file():
    raise FileNotFoundError(f"Set TEST_REPORT to the new 24-sample combined report: {TEST_REPORT}")

train_report = pd.read_excel(TRAIN_REPORT, sheet_name="Per-Sample Values")
test_report = pd.read_excel(TEST_REPORT, sheet_name="Per-Sample Values")
feature_columns = [column for column, _ in PREDICTORS]
target_columns = [column for column, _ in RESPONSES]
required_columns = ["Sample", *feature_columns, *target_columns]

for path, frame in [(TRAIN_REPORT, train_report), (TEST_REPORT, test_report)]:
    missing = set(required_columns) - set(frame.columns)
    if missing:
        raise ValueError(f"{path.name} is missing columns: {', '.join(sorted(missing))}")

if len(train_report) != 55:
    raise ValueError(f"Expected the original 55-sample report in TRAIN_REPORT, found {len(train_report)} rows.")
if len(test_report) != 24:
    raise ValueError(f"Expected 24 sample rows in {TEST_REPORT.name}, found {len(test_report)}.")

test_values = test_report[feature_columns + target_columns].apply(pd.to_numeric, errors="coerce")
nonfinite = ~np.isfinite(test_values)
invalid = nonfinite.any(axis=1)
if invalid.any():
    issues = []
    for row in test_values.index[invalid]:
        fields = [
            f"{column}={test_report.at[row, column]!r} (missing, nonnumeric, or non-finite)"
            for column in test_values.columns
            if nonfinite.at[row, column]
        ]
        issues.append(f"  {test_report.at[row, 'Sample']}: {', '.join(fields)}")
    raise ValueError(
        "Check these fields in the test report's Per-Sample Values sheet:\n" + "\n".join(issues)
    )

x_test = test_values[feature_columns].to_numpy()
all_test_rows = np.ones(len(test_values), dtype=bool)
power_sum_rows = (x_test > 0).all(axis=1)
test_predictions = test_report[required_columns].copy()
test_predictions["Power-sum exclusion reason"] = ""
for row in np.flatnonzero(~power_sum_rows):
    fields = [
        f"{column}={x_test[row, i]:g}"
        for i, column in enumerate(feature_columns)
        if x_test[row, i] <= 0
    ]
    test_predictions.loc[row, "Power-sum exclusion reason"] = ", ".join(fields)

excluded = test_predictions.loc[~power_sum_rows, ["Sample", "Power-sum exclusion reason"]]
if not excluded.empty:
    print(f"Power-sum can be evaluated on {power_sum_rows.sum()} of {len(test_values)} samples.")
    
test_scores = []


def score_test_predictions(target_name, model_name, test_set, actual, predicted, train_actual):
    error = actual - predicted
    sse = np.sum(error ** 2)
    test_sst = np.sum((actual - actual.mean()) ** 2)
    return {
        "Image metric": target_name,
        "Model": model_name,
        "Test set": test_set,
        "Training samples": len(train_actual),
        "Test samples": len(actual),
        "RMSE": np.sqrt(np.mean(error ** 2)),
        "MAE": np.mean(np.abs(error)),
        "Mean residual (actual - predicted)": np.mean(error),
        "Test R²": 1 - sse / test_sst if test_sst > 0 else np.nan,
        "Training-mean baseline RMSE": np.sqrt(np.mean((actual - train_actual.mean()) ** 2)),
    }


for target, target_name in RESPONSES:
    train = train_report[feature_columns + [target]].dropna()
    x_train = train[feature_columns].to_numpy(dtype=float)
    y_train = train[target].to_numpy(dtype=float)
    if len(y_train) <= len(feature_columns):
        raise ValueError(f"Too few complete training samples for {target_name}.")
    if not np.isfinite(x_train).all() or not np.isfinite(y_train).all() or (x_train <= 0).any():
        raise ValueError(f"Check the training values for {target_name}. Optical predictors must be finite and positive.")

    y_test = test_values[target].to_numpy()

    for model_name, fit_model in MODELS:
        prediction_rows = power_sum_rows if model_name == "Power-sum" else all_test_rows
        prediction = np.full(len(test_values), np.nan)
        if prediction_rows.any():
            fit = fit_model(x_train, y_train)
            if fit is None:
                raise RuntimeError(f"{model_name} fit failed for {target_name}.")
            prediction[prediction_rows] = fit["predict"](x_test[prediction_rows])
            if not np.isfinite(prediction[prediction_rows]).all():
                raise ValueError(f"{model_name} gave non-finite predictions for {target_name}.")

        test_predictions[f"{target} {model_name} predicted"] = prediction
        test_predictions[f"{target} {model_name} residual"] = y_test - prediction

        if model_name == "Linear" or power_sum_rows.all():
            test_scores.append(score_test_predictions(
                target_name, model_name, "All test samples", y_test, prediction, y_train
            ))
        if power_sum_rows.any() and not power_sum_rows.all():
            test_scores.append(score_test_predictions(
                target_name, model_name, "Positive predictors",
                y_test[power_sum_rows], prediction[power_sum_rows], y_train
            ))

test_summary = pd.DataFrame(test_scores)
with pd.ExcelWriter(TEST_OUTPUT, engine="openpyxl") as writer:
    test_summary.to_excel(writer, sheet_name="Test metrics", index=False)
    test_predictions.to_excel(writer, sheet_name="Per-sample predictions", index=False)
    pd.DataFrame({"Training report": [str(TRAIN_REPORT)], "Test report": [str(TEST_REPORT)]}).to_excel(
        writer, sheet_name="Inputs", index=False
    )

print(test_summary.to_string(index=False))
print(f"Saved validation report to {TEST_OUTPUT}")

### Read the validation workbook

- **Test metrics** gives RMSE, MAE, mean residual, and test R². When the power-sum model covers fewer rows, compare the two models using the **Positive predictors** scores. The training-mean baseline predicts the original training average for each test sample.
- **Per-sample predictions** includes all 24 new samples. A positive residual means the model underestimated the measured image metric. A blank power-sum prediction has a corresponding reason in the same row.
- **Inputs** records the source report paths.